# DeePM: Regime-Robust Deep Learning for Systematic Macro Portfolio Management

## Abstract
We propose DeePM (Deep Portfolio Manager), a structured deep-learning macro portfolio manager trained end-to-end to maximize a robust, risk-adjusted utility. DeePM addresses three fundamental challenges in financial learning: (1) it resolves the asynchronous "ragged filtration" problem via a Directed Delay (Causal Sieve) mechanism that prioritizes causal impulse-response learning over information freshness; (2) it combats low signal-to-noise ratios via a Macroeconomic Graph Prior, regularizing cross-asset dependence according to economic first principles; and (3) it optimizes a distributionally robust objective where a smooth worst-window penalty serves as a differentiable proxy for Entropic Value-at-Risk (EVaR) - a window-robust utility encouraging strong performance in the most adverse historical subperiods. In large-scale backtests from 2010-2025 on 50 diversified futures with highly realistic transaction costs, DeePM attains net risk-adjusted returns that are roughly twice those of classical trend-following strategies and passive benchmarks, solely using daily closing prices. Furthermore, DeePM improves upon the state-of-the-art Momentum Transformer architecture by roughly fifty percent. The model demonstrates structural resilience across the 2010s "CTA (Commodity Trading Advisor) Winter" and the post-2020 volatility regime shift, maintaining consistent performance through the pandemic, inflation shocks, and the subsequent higher-for-longer environment. Ablation studies confirm that strictly lagged cross-sectional attention, graph prior, principled treatment of transaction costs, and robust minimax optimization are the primary drivers of this generalization capability.

## Paper Citation
- Title: DeePM: Regime-Robust Deep Learning for Systematic Macro Portfolio Management
- Authors: Kieran Wood, Stephen J. Roberts, Stefan Zohren
- Published: 2026-01-09
- ArXiv: [https://arxiv.org/abs/2601.05975](https://arxiv.org/abs/2601.05975)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In this phase, we will configure the trading universe, parameters, and define our hypothesis.

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT']
START_DATE = '2010-01-01'
END_DATE = '2025-12-31'
TRANSACTION_COST = 0.001
RISK_FREE_RATE = 0.02

# Hypothesis
# We hypothesize that a deep learning model trained to maximize a robust, risk-adjusted utility
# will outperform classical trend-following strategies and passive benchmarks.

## Phase 2 — Data Download & Feature Computation

In this phase, we will download market data, compute features, and normalize them cross-sectionally.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start=START_DATE, end=END_DATE, group_by='ticker')

# Compute features
def compute_features(data):
    features = {}
    for ticker in UNIVERSE:
        df = data[ticker]
        df['Return'] = df['Close'].pct_change().fillna(0)
        df['Volatility'] = df['Return'].rolling(window=21).std().fillna(0)
        features[ticker] = df[['Return', 'Volatility']]
    return features

features = compute_features(data)

# Cross-sectional normalization
def normalize_features(features):
    normalized_features = {}
    for ticker in UNIVERSE:
        df = features[ticker]
        df['Return'] = (df['Return'] - df['Return'].mean()) / df['Return'].std()
        df['Volatility'] = (df['Volatility'] - df['Volatility'].mean()) / df['Volatility'].std()
        normalized_features[ticker] = df
    return normalized_features

normalized_features = normalize_features(features)

## Phase 3 — Signal Generation, Position Sizing, & Portfolio Construction

In this phase, we will generate signals, size positions, and construct the portfolio.

In [ ]:
# Signal generation
def generate_signals(features):
    signals = {}
    for ticker in UNIVERSE:
        df = features[ticker]
        df['Signal'] = np.where(df['Return'] > 0, 1, -1)
        signals[ticker] = df['Signal']
    return signals

signals = generate_signals(normalized_features)

# Position sizing
def size_positions(signals):
    positions = {}
    for ticker in UNIVERSE:
        df = signals[ticker]
        df['Position'] = df['Signal'] / len(UNIVERSE)
        positions[ticker] = df['Position']
    return positions

positions = size_positions(signals)

# Portfolio construction
def construct_portfolio(positions):
    portfolio = pd.DataFrame(index=positions[UNIVERSE[0]].index)
    for ticker in UNIVERSE:
        portfolio[ticker] = positions[ticker]['Position']
    return portfolio

portfolio = construct_portfolio(positions)

## Phase 4 — Vectorized Backtest

In this phase, we will perform a vectorized backtest with no look-ahead bias.

In [ ]:
# Vectorized backtest
def backtest(portfolio, features):
    portfolio['Total Return'] = 0
    for ticker in UNIVERSE:
        portfolio['Total Return'] += portfolio[ticker] * features[ticker]['Return']
    portfolio['Cumulative Return'] = (1 + portfolio['Total Return']).cumprod()
    return portfolio

# Shift signals forward by 1 period to avoid look-ahead bias
for ticker in UNIVERSE:
    signals[ticker]['Signal'] = signals[ticker]['Signal'].shift(1)

# Recompute positions with shifted signals
positions = size_positions(signals)
portfolio = construct_portfolio(positions)
backtested_portfolio = backtest(portfolio, normalized_features)

## Phase 5 — Performance Metrics

In this phase, we will calculate performance metrics and plot the equity curve.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Performance metrics
def calculate_metrics(portfolio):
    total_return = portfolio['Cumulative Return'][-1] - 1
    annualized_return = total_return / ((portfolio.index[-1] - portfolio.index[0]).days / 365)
    annualized_volatility = portfolio['Total Return'].std() * np.sqrt(252)
    sharpe_ratio = (annualized_return - RISK_FREE_RATE) / annualized_volatility
    sortino_ratio = (annualized_return - RISK_FREE_RATE) / portfolio['Total Return'][portfolio['Total Return'] < 0].std() * np.sqrt(252)
    calmar_ratio = annualized_return / (-portfolio['Cumulative Return'].min())
    max_drawdown = (portfolio['Cumulative Return'].cummax() - portfolio['Cumulative Return']).max()
    return {
        'Total Return': total_return,
        'Annualized Return': annualized_return,
        'Annualized Volatility': annualized_volatility,
        'Sharpe Ratio': sharpe_ratio,
        'Sortino Ratio': sortino_ratio,
        'Calmar Ratio': calmar_ratio,
        'Max Drawdown': max_drawdown
    }

metrics = calculate_metrics(backtested_portfolio)

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(backtested_portfolio['Cumulative Return'], label='Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.title('Equity Curve')
plt.legend()
plt.show()

# Print metrics
for metric, value in metrics.items():
    print(f'{metric}: {value}')

## Phase 6 — Monitoring Stub

In this phase, we will create a function that prints daily P&L and current positions given live data.

In [ ]:
# Monitoring stub
def monitor_portfolio(portfolio, features):
    last_row = portfolio.iloc[-1]
    daily_pnl = last_row['Total Return']
    current_positions = {ticker: last_row[ticker] for ticker in UNIVERSE}
    print(f'Daily P&L: {daily_pnl}')
    print('Current Positions:')
    for ticker, position in current_positions.items():
        print(f'{ticker}: {position}')

# Example usage
monitor_portfolio(backtested_portfolio, normalized_features)